## 熵权法 + TOPSIS（Python 3.11）

**一、熵权法：让数据自己"投票"决定权重**

1. **正向化**：把极小型、中间型、区间型指标统一转成"越大越好"
2. **标准化**：消除量纲 $z_{ij}=\dfrac{x_{ij}}{\sqrt{\sum_i x_{ij}^2}}$
3. **计算比重**：$p_{ij}=\dfrac{z_{ij}}{\sum_i z_{ij}}$
4. **信息熵**：$e_j=-\dfrac{1}{\ln n}\sum_i p_{ij}\ln p_{ij}$（约定 $0\ln 0=0$）
5. **熵权**：$w_j=\dfrac{1-e_j}{\sum_k (1-e_k)}$ —— 指标差异越大，信息越多，权重越大

**二、TOPSIS：离正理想解越近、离负理想解越远越好**

1. 加权规范化矩阵 $v_{ij}=w_j z_{ij}$
2. 正理想解 $v_j^+=\max_i v_{ij}$，负理想解 $v_j^-=\min_i v_{ij}$
3. 欧氏距离 $D_i^{\pm}=\sqrt{\sum_j (v_{ij}-v_j^{\pm})^2}$
4. 贴近度 $C_i=\dfrac{D_i^-}{D_i^+ + D_i^-}\in[0,1]$，越大越优

**三、指标正向化公式**

| 类型 | 含义 | 正向化公式 |
| --- | --- | --- |
| `benefit` | 极大型（效益型） | $y=x$ |
| `cost` | 极小型（成本型） | $y=\max(x)-x$ |
| `intermediate` | 中间型（最优值 $x_{best}$） | $y=1-\dfrac{|x-x_{best}|}{\max|x-x_{best}|}$ |
| `interval` | 区间型（最优区间 $[a,b]$） | $y=1-\dfrac{d(x,[a,b])}{\max d(x,[a,b])}$ |


### 1. 环境准备

In [1]:
# ============================================================
# 本 Notebook 用「熵权法（EWM）+ TOPSIS」做综合评价：
#   熵权法  —— 让数据自己"投票"，根据指标内部的差异程度自动算出权重
#   TOPSIS —— 计算每个评价对象离"理想解"有多远，给出综合得分与排名
# 运行环境：Python 3.11
# ============================================================
import sys
import numpy as np                 # 数值计算：矩阵、广播、范数等
import pandas as pd                # 表格展示：DataFrame 便于对齐、排序、看结果
import matplotlib.pyplot as plt    # 画图：权重条形图 + 得分柱状图

# 指定中文字体，避免图表里的中文显示成方块
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]
# 允许正常显示负号（否则坐标轴的负号也可能变成方块）
plt.rcParams["axes.unicode_minus"] = False
# pandas 打印时按中文全角宽度对齐，表格更整齐
pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 120)

# 打印当前 Python 版本，确认环境是 3.11
print("Python 版本：", sys.version.split()[0])

Python 版本： 3.11.9


### 2. 指标正向化

In [ ]:
# 四种指标类型（正向化的四种公式）：
#   benefit      极大型（效益型）：数值越大越好，例如 GDP、教育投入
#   cost         极小型（成本型）：数值越小越好，例如能耗、污染排放
#   intermediate 中间型：越接近某个最优值越好，例如最适宜气温 22℃
#   interval     区间型：落在某个最优区间内最好，例如绿化覆盖率 40%~50%
KINDS = ("benefit", "cost", "intermediate", "interval")


def positivize(X, kinds, params=None):
    """把所有指标统一正向化为"越大越好"。

    X      : (n_samples, m_indicators) 原始数据（行=样本，列=指标）
    kinds  : 长度 m 的列表，元素取自 KINDS，说明每一列的类型
    params : 与 kinds 一一对应的参数
             intermediate -> x_best（最优值，一个数）
             interval     -> (a, b)（最优区间的左右端点）
             其他类型不需要参数，填 None 即可
    返回正向化后的矩阵（与 X 同形状，每列都满足"越大越好"）
    """
    # 统一转成 float 数组，避免整数除法、类型不一致等隐患
    X = np.asarray(X, dtype=float)

    # ---------- 先做一组参数校验：出错时给出明确提示，方便排查 ----------
    if X.ndim != 2:                       # 必须二维：行是样本，列是指标
        raise ValueError("X 必须是二维矩阵（样本 × 指标）")
    if len(kinds) != X.shape[1]:          # 每个指标都要说明自己的类型
        raise ValueError("kinds 长度必须等于指标个数")
    if not set(kinds) <= set(KINDS):      # 类型只能是上面那四种之一
        raise ValueError(f"指标类型只能是 {KINDS}")

    # params 允许不传，不传就当成"全为 None"
    params = list(params) if params is not None else [None] * len(kinds)
    if len(params) != len(kinds):
        raise ValueError("params 长度必须等于指标个数")

    # 结果矩阵：先复制原始数据，后面逐列覆盖成正向化后的值
    Y = X.copy()
    # 一次算出每一列的最大值（成本型公式 y = max(x) - x 需要用到）
    col_max = X.max(axis=0)

    # 逐列（逐个指标）处理：j 是列号，kind 是该列的类型
    for j, kind in enumerate(kinds):
        col = X[:, j]                     # 取出第 j 列（所有样本在该指标上的取值）

        if kind == "benefit":
            # 极大型：本来就越大越好，原样保留
            Y[:, j] = col

        elif kind == "cost":
            # 极小型：用「该列最大值 - 原值」把方向翻转，原来越小现在越大
            Y[:, j] = col_max[j] - col

        elif kind == "intermediate":
            # 中间型：离最优值 x_best 越近越好
            # gap = |x - x_best| 是偏差，再把偏差除以最大偏差压到 [0, 1]
            x_best = float(params[j])
            gap = np.abs(col - x_best)
            top = gap.max()
            # top == 0 表示所有样本都恰好等于最优值，此时统一给满分 1
            Y[:, j] = 1.0 - gap / top if top > 0 else 1.0

        else:  # interval：区间型，落在 [a, b] 之内最好
            a, b = params[j]
            lo, hi = min(a, b), max(a, b)   # 防止把区间端点写反
            below = np.where(col < lo, lo - col, 0.0)   # 低于下界的距离（区间内算 0）
            above = np.where(col > hi, col - hi, 0.0)   # 高于上界的距离（区间内算 0）
            dist = below + above                        # 到区间的最短距离
            top = dist.max()
            # 距离为 0 → 得分 1（正好在区间内）；距离最大 → 得分 0
            Y[:, j] = 1.0 - dist / top if top > 0 else 1.0

    # 返回的矩阵每一列都是"越大越好"，可以直接交给熵权法
    return Y

### 3. 熵权法求权重

In [ ]:
def entropy_weight(X):
    """熵权法求权重，要求 X 已正向化且非负。

    返回 (w, Z, e)：
      w —— 各指标权重，所有列权重之和为 1
      Z —— 向量归一化后的矩阵，供 TOPSIS 继续使用
      e —— 各指标信息熵（越接近 1 说明该指标在各样本间差异越小、权重越低）
    """
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X 必须是二维矩阵")
    if (X < 0).any():
        # 熵权法里要算 p * ln(p)，要求数据非负，所以必须先正向化
        raise ValueError("熵权法要求数据非负，请先用 positivize 正向化")
    n, m = X.shape          # n = 样本个数（如城市数），m = 指标个数
    if n < 2:
        raise ValueError("样本数至少为 2")

    # 1. 向量归一化：每列除以该列的"向量长度"，消除不同量纲带来的影响
    #    np.linalg.norm(X, axis=0) 等价于对每一列求 sqrt(Σ x²)
    #    where 用来防止整列全为 0 时出现除以 0（这种情况直接除以 1）
    norm = np.linalg.norm(X, axis=0)
    Z = X / np.where(norm == 0, 1.0, norm)

    # 2. 计算比重 p_ij：第 j 个指标下，第 i 个样本占该指标总和的比例
    #    处理完之后，每一列加起来都等于 1
    col_sum = Z.sum(axis=0)
    P = Z / np.where(col_sum == 0, 1.0, col_sum)

    # 3. 信息熵 e_j = -(1/ln n) * Σ p_ij * ln(p_ij)
    #    约定 0 * ln 0 = 0，所以先用 where 把 p == 0 的位置替换成 0 再求和
    #    直觉：某个指标各样本取值越平均 → 熵越大 → 提供的信息越少 → 权重应越小
    with np.errstate(divide="ignore", invalid="ignore"):
        L = np.where(P > 0, P * np.log(P), 0.0)
    e = -L.sum(axis=0) / np.log(n)

    # 4. 差异系数 d = 1 - e：熵越小（各样本差异越大）→ d 越大 → 最终权重越大
    #    再归一化，让所有指标的权重之和等于 1
    d = 1.0 - e
    total = d.sum()
    # 极端情况：所有列取值完全相同 → 每个 e 都是 1 → d 全为 0，此时退化为等权
    w = d / total if total > 0 else np.full(m, 1.0 / m)
    return w, Z, e

### 4. TOPSIS 综合评价

In [ ]:
def topsis(Z, w):
    """加权 TOPSIS，输入为标准化矩阵 Z 与权重 w。

    返回 (C, d_pos, d_neg, v_best, v_worst)：
      C        —— 贴近度，取值 [0, 1]，越大说明越接近理想方案
      d_pos    —— 到正理想解的距离
      d_neg    —— 到负理想解的距离
      v_best   —— 正理想解（每个指标的最优值）
      v_worst  —— 负理想解（每个指标的最差值）
    """
    Z = np.asarray(Z, dtype=float)
    w = np.asarray(w, dtype=float)
    if Z.ndim != 2 or Z.shape[1] != w.size:
        raise ValueError("Z 的列数必须与权重个数一致")

    # 1. 加权：每个指标乘上自己的权重，重要指标对结果影响更大
    V = Z * w                                       # 加权规范化矩阵
    # 2. 构造理想解：每一列的最大值就是该指标的"最好水平"，最小值是"最差水平"
    v_best = V.max(axis=0)                          # 正理想解 v+
    v_worst = V.min(axis=0)                         # 负理想解 v-

    # 3. 每个样本到正/负理想解的欧氏距离（先按行求平方和，再开根号）
    d_pos = np.sqrt(((V - v_best) ** 2).sum(axis=1))
    d_neg = np.sqrt(((V - v_worst) ** 2).sum(axis=1))

    # 4. 贴近度 C = d_neg / (d_pos + d_neg)：越靠近正理想解，C 越接近 1
    #    where 防止分母为 0 时抛出除零错误
    denom = d_pos + d_neg
    C = np.divide(d_neg, denom, out=np.zeros_like(d_neg), where=denom > 0)

    # 退化情形：正负理想解重合（所有方案完全相同）时，一律算满分 1
    C = np.where(denom > 0, C, 1.0)
    return C, d_pos, d_neg, v_best, v_worst


def ewm_topsis(X, kinds, params=None, names=None, index=None, show=True):
    """熵权法 + TOPSIS 一体化评价，返回结果 DataFrame（按贴近度降序）。

    内部流程一共三步：正向化 → 熵权法定权重 → TOPSIS 打分排名
    """
    X = np.asarray(X, dtype=float)
    m = X.shape[1]      # 指标个数
    # 没传指标名就用「指标1、指标2……」占位
    names = list(names) if names is not None else [f"指标{j + 1}" for j in range(m)]

    # 第 1 步：正向化，把所有指标统一成"越大越好"
    Y = positivize(X, kinds, params)
    # 第 2 步：熵权法，得到权重 w、标准化矩阵 Z、信息熵 e
    w, Z, e = entropy_weight(Y)
    # 第 3 步：TOPSIS，得到每个样本的贴近度 C 与到两个理想解的距离
    C, d_pos, d_neg, _, _ = topsis(Z, w)

    if show:
        # 把权重相关的中间结果打印出来，方便检查"某指标为什么权重高/低"
        table = pd.DataFrame({
            "指标": names,
            "类型": kinds,
            "信息熵 e": e,
            "差异系数 1-e": 1.0 - e,
            "权重 w": w,
        }).set_index("指标")
        print("=== 熵权法权重 ===\n")
        print(table.round(4).to_string())

    # 组装最终结果表：index 是样本名称（如城市名）
    result = pd.DataFrame({
        "贴近度 C": C,
        "正理想解距离": d_pos,
        "负理想解距离": d_neg,
    }, index=index)
    # rank(ascending=False) 让得分最高的排第 1；method="min" 表示并列时取较小名次
    result["排名"] = result["贴近度 C"].rank(ascending=False, method="min").astype(int)
    # 按贴近度从高到低排序，第一名排在最上面
    return result.sort_values("贴近度 C", ascending=False)

### 5. 示例：5 个城市的宜居性评价

指标（均为常见公开统计数据量级的模拟值）：

| 指标 | 类型 | 说明 |
| --- | --- | --- |
| 人均 GDP（万元） | 极大型 | 越高越好 |
| 单位 GDP 能耗（吨标煤/万元） | 极小型 | 越低越好 |
| 年平均气温（℃） | 中间型 | 最优 22℃ |
| 绿化覆盖率（%） | 区间型 | 最优 40%~50% |
| 教育投入占比（%） | 极大型 | 越高越好 |


In [ ]:
# 5 个待评价的城市（也就是 5 个"样本"）
cities = ["北京", "上海", "广州", "成都", "拉萨"]
# 5 个评价指标（列名），顺序与下面 X 的列顺序严格对应
indicators = ["人均GDP", "单位GDP能耗", "年平均气温", "绿化覆盖率", "教育投入占比"]

# 原始数据矩阵：每一行一个城市，每一列一个指标
#   第 1 列 人均GDP      —— 极大型，越大越好
#   第 2 列 单位GDP能耗  —— 极小型，越小越好
#   第 3 列 年平均气温    —— 中间型，最理想是 22℃
#   第 4 列 绿化覆盖率    —— 区间型，落在 40%~50% 最好
#   第 5 列 教育投入占比  —— 极大型，越大越好
X = np.array([
    [20.0, 0.30, 12.9, 48.5, 5.6],   # 北京
    [18.5, 0.35, 16.6, 39.6, 4.8],   # 上海
    [16.0, 0.40, 22.4, 43.2, 4.5],   # 广州
    [10.5, 0.45, 16.2, 44.9, 5.1],   # 成都
    [7.2,  0.25,  8.5, 38.0, 6.2],   # 拉萨
], dtype=float)

# 与 indicators 逐列对应的指标类型
kinds = ["benefit", "cost", "intermediate", "interval", "benefit"]
# 与 kinds 逐列对应的额外参数：中间型填最优值 22.0，区间型填 (40.0, 50.0)，
# 其他类型不需要参数，填 None 占位
params = [None, None, 22.0, (40.0, 50.0), None]

# 转成 DataFrame 展示：行=城市，列=指标，最直观
df = pd.DataFrame(X, index=cities, columns=indicators)
df

In [ ]:
# 看看正向化之后的数据：所有列都变成"数值越大越好"，
# 而且不同指标的量级也被拉到接近的范围
Y = positivize(X, kinds, params)
pd.DataFrame(Y.round(4), index=cities, columns=indicators)

In [ ]:
# 一步到位：正向化 → 熵权法 → TOPSIS，直接得到每个城市的综合得分与排名
result = ewm_topsis(X, kinds, params, names=indicators, index=cities)
print("\n=== 综合评价结果（按贴近度降序）===\n")
result.round(4)

### 6. 结果可视化

In [ ]:
# 重新求一次权重（Y 就是上面已经正向化的矩阵），画图时要用
w, Z, e = entropy_weight(Y)

# 画布：1 行 2 列，左边放各指标权重，右边放各城市得分
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# ---------- 左图：各指标权重 ----------
order = np.argsort(w)   # 返回权重从小到大的列索引，让条形图从上到下递增更好读
axes[0].barh(np.array(indicators)[order], w[order], color="#4C72B0")
axes[0].set_title("熵权法权重")
axes[0].set_xlabel("权重")
for k, idx in enumerate(order):
    # 在每根条形右侧标上具体数值，保留 3 位小数
    axes[0].text(w[idx], k, f" {w[idx]:.3f}", va="center", fontsize=9)

# ---------- 右图：各城市 TOPSIS 贴近度 ----------
score = result["贴近度 C"]   # result 已按贴近度降序排好
axes[1].bar(score.index, score.values, color="#55A868")
axes[1].set_title("TOPSIS 贴近度排名")
axes[1].set_ylabel("贴近度 C")
axes[1].set_ylim(0, 1)       # 贴近度固定落在 [0, 1]，统一纵轴范围便于比较
for name, value in score.items():
    # 在柱子顶部标上具体得分
    axes[1].text(name, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()   # 自动调整子图间距，避免标签被裁掉
plt.show()